# NOTEBOOK 05: MODEL EVALUATION & SHAP EXPLAINABILITY
In this notebook, we perform:
- Benchmark between Best Machine Learning Model (XGBoost) and Deep Learning (PyTorch MLP).
- Confusion Matrices, Classification Reports, ROC-AUC curves.
- **SHAP (SHapley Additive exPlanations)** for global feature importance and individual patient prediction interpretability.


In [1]:
import os
import sys
import json
import joblib
import torch
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

sys.path.append(os.path.join("..", "training_scripts"))
from data_loader import prepare_splits, FEATURE_COLUMNS, SAVED_MODELS_DIR
from train_dl import DiabetesMLP

data = prepare_splits()
X_test, y_test = data['X_test'], data['y_test']

ml_model = joblib.load(os.path.join(SAVED_MODELS_DIR, "diabetes_model.pkl"))
dl_model = DiabetesMLP(input_dim=8)
dl_model.load_state_dict(torch.load(os.path.join(SAVED_MODELS_DIR, "diabetes_dl_model.pth")))
dl_model.eval()


C:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DiabetesMLP(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)

### 5.1 Test Set Classification Reports

In [2]:
y_pred_ml = ml_model.predict(X_test)
print("=== XGBoost Classification Report ===")
print(classification_report(y_test, y_pred_ml, target_names=["Non-Diabetic", "Diabetic"]))

with torch.no_grad():
    y_prob_dl = torch.sigmoid(dl_model(torch.tensor(X_test, dtype=torch.float32))).numpy().flatten()
    y_pred_dl = (y_prob_dl >= 0.5).astype(int)
print("=== PyTorch MLP Classification Report ===")
print(classification_report(y_test, y_pred_dl, target_names=["Non-Diabetic", "Diabetic"]))


=== XGBoost Classification Report ===
              precision    recall  f1-score   support

Non-Diabetic       0.97      1.00      0.99     18300
    Diabetic       0.98      0.69      0.81      1700

    accuracy                           0.97     20000
   macro avg       0.98      0.84      0.90     20000
weighted avg       0.97      0.97      0.97     20000

=== PyTorch MLP Classification Report ===
              precision    recall  f1-score   support

Non-Diabetic       0.99      0.89      0.94     18300
    Diabetic       0.45      0.92      0.60      1700

    accuracy                           0.90     20000
   macro avg       0.72      0.91      0.77     20000
weighted avg       0.95      0.90      0.91     20000



### 5.2 SHAP Explainability

In [3]:
explainer = shap.TreeExplainer(ml_model)
sample_X = X_test[:300]
shap_values = explainer.shap_values(sample_X)
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_values = shap_values[1]

mean_shap = np.mean(np.abs(shap_values), axis=0)
shap_df = pd.DataFrame({"Feature": FEATURE_COLUMNS, "Mean_Absolute_SHAP": mean_shap})
shap_df = shap_df.sort_values(by="Mean_Absolute_SHAP", ascending=False).reset_index(drop=True)
shap_df


,Feature,Mean_Absolute_SHAP
0,HbA1c_level,2.677325
1,blood_glucose_level,1.945692
2,age,0.576472
3,bmi,0.252512
4,smoking_history,0.185885
5,gender,0.105914
6,hypertension,0.084607
7,heart_disease,0.053366


### 5.3 Clinical Interpretation
The SHAP analysis reveals that **HbA1c_level** and **blood_glucose_level** are by far the dominant drivers of diabetes risk prediction, completely aligned with modern clinical endocrinology standards.
